# 01. 프롬프트 엔지니어링 실습

---

## 학습 목표

LLM의 출력 품질을 높이는 **핵심 프롬프트 기법 6가지**를 실습합니다.  
같은 질문에 다른 기법을 적용해 결과 차이를 직접 확인합니다.

## 다루는 기법

| 기법 | 핵심 아이디어 | 언제 쓰나? |
|---|---|---|
| **Zero-shot** | 예시 없이 바로 질문 | 간단한 일반 질문 |
| **Few-shot** | 예시(샷)를 프롬프트에 포함 | 특정 형식·스타일 유도 |
| **Chain-of-Thought** | 단계별 추론 유도 | 복잡한 분석·판단 |
| **역할 프롬프트** | LLM에게 페르소나 부여 | 전문 도메인 답변 |
| **구조화된 출력** | 특정 형식(JSON 등) 강제 | 파싱이 필요한 경우 |
| **프롬프트 템플릿** | 변수로 재사용 가능한 프롬프트 | 반복 작업 자동화 |

## 흐름

```
기법 학습 → 코드 실습 → 같은 질문으로 비교 실험 → 핵심 정리
```

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json
import sys
sys.path.append("..")

from common_config import llm_connect
model ="gpt-5.4"
temperature = 0
max_tokens = 2086

# LLM 객체 생성
llm = llm_connect(model=model)
# llm = llm_connect(model=model, temperature=temperature, max_tokens=max_tokens)


print("준비 완료")
print("실습에 사용할 공통 질문: '재택근무 전면 도입, 찬성해야 할까요?'")

준비 완료
실습에 사용할 공통 질문: '재택근무 전면 도입, 찬성해야 할까요?'


---

## 1. Zero-shot Prompting

**예시 없이** 바로 질문하는 가장 기본적인 방식입니다.  
LLM이 사전 학습 지식만으로 답변합니다.

```
사용자: [질문]
LLM:   [답변]
```

In [2]:
# ── Zero-shot: 예시 없이 바로 질문 ────────────────────────────
zero_shot_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

# 체인 생성 및 실행
chain = zero_shot_prompt | llm | StrOutputParser()
result = chain.invoke({"question": "재택근무 전면 도입, 찬성해야 할까요?"})

print("=== Zero-shot 결과 ===")
print(result)

=== Zero-shot 결과 ===
재택근무 전면 도입은 **무조건 찬성/반대**로 보기보다, **조직의 성격에 따라 신중히 판단해야 하는 주제**입니다.  
다만 질문이 “찬성해야 할까요?”라면, 저는 **조건부 찬성** 쪽입니다.

### 찬성하는 이유
1. **업무 효율 향상 가능성**
   - 출퇴근 시간이 줄어들어 직원 피로가 감소합니다.
   - 집중이 중요한 업무는 사무실보다 집에서 더 잘되는 경우가 많습니다.

2. **직원 만족도 증가**
   - 일과 삶의 균형이 좋아질 수 있습니다.
   - 육아, 건강, 거주지 문제 등 다양한 상황을 가진 직원들에게 유리합니다.

3. **기업 비용 절감**
   - 사무실 임대료, 관리비, 운영비를 줄일 수 있습니다.

4. **인재 확보에 유리**
   - 지역 제한 없이 인재를 채용할 수 있습니다.
   - 유연한 근무제는 우수 인재에게 매력적입니다.

### 반대하거나 주의해야 할 이유
1. **협업과 소통 문제**
   - 즉각적인 의사소통이 어려워질 수 있습니다.
   - 창의적 아이디어 회의나 빠른 협업은 대면이 더 유리할 수 있습니다.

2. **조직문화 약화**
   - 소속감, 동료애, 회사 문화 형성이 어려워질 수 있습니다.
   - 신입 직원은 적응에 더 큰 어려움을 겪을 수 있습니다.

3. **업무 관리의 어려움**
   - 성과 측정 체계가 명확하지 않으면 생산성이 떨어질 수 있습니다.
   - 일부 직원은 자기관리 부담이 커질 수 있습니다.

4. **업무와 사생활의 경계 모호**
   - 오히려 근무 시간이 길어지거나 스트레스가 증가할 수 있습니다.

### 결론
따라서 재택근무 전면 도입은 **장점이 크지만, 모든 조직에 일괄적으로 적용하기엔 위험**합니다.  
가장 현실적인 입장은 **전면 도입 자체에 찬성하되, 직무와 조직 특성에 따라 보완책을 마련해야 한다**는 것입니다.  
�


---

## 2. Few-shot Prompting

프롬프트 안에 **예시(shots)** 를 포함해서 원하는 출력 형식을 보여줍니다.  
LLM이 패턴을 학습해 같은 형식으로 답변합니다.

```
시스템: 예시1: Q→A, 예시2: Q→A  ← 패턴 학습
사용자: [새 질문]
LLM:   [예시와 같은 형식의 답변]  ← 패턴 따라함
```

In [3]:
# ── Few-shot: 예시 2개를 포함해서 출력 형식을 유도 ────────────
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", """아래 예시와 똑같은 형식으로 답변하세요.

예시 1)
질문: 주 4일 근무제를 도입해야 할까요?
답변:
✅ 찬성 근거
- 직원 삶의 질 향상으로 생산성 증가
- 채용 경쟁력 강화

❌ 반대 근거
- 업무량 미감소 시 오히려 번아웃 위험
- 서비스 연속성 유지 어려움

💡 결론: 업종·직무별로 차등 적용하는 파일럿 테스트를 권장합니다.

예시 2)
질문: 사내 카페테리아를 없애야 할까요?
답변:
✅ 찬성 근거
- 운영 비용 절감
- 외부 다양한 식사 옵션 활용 가능

❌ 반대 근거
- 직원 식사 복지 감소
- 점심 시간 이탈로 협업 기회 감소

💡 결론: 직원 수요 조사 후 결정하되, 식대 지원으로 보완을 고려하세요."""),
    ("human", "질문: {question}")
])

# 체인 생성 및 실행
chain = few_shot_prompt | llm | StrOutputParser()
result = chain.invoke({"question": "재택근무 전면 도입, 찬성해야 할까요?"})

print("=== Few-shot 결과 ===")
print(result)

=== Few-shot 결과 ===
답변:
✅ 찬성 근거
- 출퇴근 시간 절감으로 직원 만족도와 업무 몰입도 향상
- 지역 제한 없는 채용으로 인재 확보 범위 확대
- 사무실 운영비 절감 가능

❌ 반대 근거
- 협업·소통 저하로 의사결정 속도 감소 가능
- 조직문화 약화 및 소속감 저하 우려
- 보안 관리와 업무 통제의 어려움 증가

💡 결론: 전면 도입보다는 직무별 특성을 반영한 하이브리드 방식부터 적용하는 것이 현실적입니다.


---

## 3. Chain-of-Thought (CoT) Prompting

LLM에게 **단계별로 생각하도록** 유도합니다.  
복잡한 추론, 계획, 분석이 필요한 질문에 효과적입니다.

```
기본:  질문 → 바로 답변 (shallow)
CoT:  질문 → 단계1 → 단계2 → 단계3 → 결론 (deep)
```

In [4]:
# ── Chain-of-Thought: 단계별 사고 과정 명시 ───────────────────
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", """질문에 답하기 전에 반드시 다음 4단계로 생각하고, 각 단계를 명시하세요.

[1단계: 문제 파악] 무엇을 묻고 있는가?
[2단계: 이해관계자 분석] 누가 영향을 받는가?
[3단계: 장단점 검토] 도입/미도입 시 각각 어떤 결과가?
[4단계: 최종 판단] 근거를 바탕으로 한 결론은?"""),
    ("human", "{question}")
])

# 체인 생성 및 실행
chain = cot_prompt | llm | StrOutputParser()
result = chain.invoke({"question": "재택근무 전면 도입, 찬성해야 할까요?"})

print("=== Chain-of-Thought 결과 ===")
print(result)

=== Chain-of-Thought 결과 ===
[1단계: 문제 파악]  
질문은 **“재택근무를 전면 도입하는 것에 찬성할 것인가”**에 대한 판단입니다.  
핵심은 단순히 재택근무가 좋은지 여부가 아니라, **모든 업무와 조직에 예외 없이 전면 적용하는 것이 타당한가**입니다.

[2단계: 이해관계자 분석]  
영향을 받는 주체는 다양합니다.

- **직원**: 출퇴근 부담 감소, 일·생활 균형 향상 가능
- **기업/경영진**: 사무실 비용 절감, 생산성 변화, 조직 관리 방식 변화
- **팀 관리자**: 성과 관리, 소통 방식, 협업 구조 재설계 필요
- **고객/거래처**: 대응 속도나 서비스 품질 변화 가능
- **사회 전반**: 교통 혼잡 감소, 상권 변화, 지역경제 영향

즉, 직원 편의만이 아니라 **조직 운영과 서비스 품질까지 함께 봐야 하는 문제**입니다.

[3단계: 장단점 검토]  

### 1. 전면 도입에 찬성할 이유
- **출퇴근 시간 절감**으로 직원 만족도가 높아질 수 있습니다.
- **사무실 운영비 절감** 효과가 있습니다.
- 집중이 필요한 업무는 오히려 **생산성이 높아질 가능성**이 있습니다.
- 지역 제약 없이 **인재 채용 범위가 넓어질 수 있습니다.**
- 교통량 감소, 탄소 저감 등 **사회적 이익**도 기대됩니다.

### 2. 전면 도입의 문제점
- 모든 직무가 재택에 적합한 것은 아닙니다.  
  예: 대면 서비스, 현장 관리, 장비 활용 업무
- 팀 내 **즉각적 소통과 협업의 질**이 떨어질 수 있습니다.
- 신입사원 교육, 조직문화 형성, 멘토링이 어려워질 수 있습니다.
- 직원에 따라 집의 업무 환경 차이가 커서 **형평성 문제**가 생길 수 있습니다.
- 장기적으로는 고립감, 소속감 약화, 성과평가 갈등이 발생할 수 있습니다.
- 보안


---

## 4. 역할(Role) 프롬프트

LLM에게 **특정 전문가 페르소나**를 부여합니다.  
같은 질문이라도 역할에 따라 완전히 다른 관점으로 답변합니다.

```
역할 없음:  중립적·일반적 답변
HR 전문가:  사람 중심의 조직 관점
CFO:       비용·효율 중심의 재무 관점
```

In [5]:
# ── 역할 프롬프트: 같은 질문, 다른 역할 ──────────────────────
def make_role_chain(role_description: str):
    """역할 설명을 받아 전문가 체인을 생성한다"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", role_description),
        ("human", "{question}")
    ])
    chain = prompt | llm | StrOutputParser()
    return chain


# 역할 1: HR 디렉터
hr_chain = make_role_chain("""
당신은 15년 경력의 HR 디렉터입니다.
직원 복지, 조직 문화, 인재 유지에 최우선 가치를 둡니다.
답변은 구성원 관점에서 3~4문장으로 핵심만 짚어주세요.
""")

# 역할 2: 재무이사(CFO)
cfo_chain = make_role_chain("""
당신은 10년 경력의 재무이사(CFO)입니다.
모든 결정을 비용-효익 분석으로 접근합니다.
답변은 수치 근거를 중심으로 3~4문장으로 핵심만 짚어주세요.
""")

question = "재택근무 전면 도입, 찬성해야 할까요?"

print("=== HR 디렉터 관점 ===")
print(hr_chain.invoke({"question": question}))

print("\n=== CFO 관점 ===")
print(cfo_chain.invoke({"question": question}))

=== HR 디렉터 관점 ===
전면 재택근무는 구성원의 자율성과 몰입도를 높이고, 출퇴근 부담을 줄여 삶의 질과 만족도를 크게 개선할 수 있습니다. 다만 협업 밀도, 소속감, 신입 온보딩처럼 조직문화에 중요한 요소가 약해질 위험도 함께 봐야 합니다. 그래서 무조건 찬반으로 보기보다, 직무 특성과 팀 운영 방식에 따라 하이브리드나 단계적 도입이 더 현실적일 수 있습니다. 핵심은 회사 편의보다 구성원이 지속적으로 잘 일할 수 있는 환경을 설계하는 데 있습니다.

=== CFO 관점 ===
전면 재택은 고정비 절감 효과가 명확합니다. 예를 들어 100명 조직 기준으로 사무실 임차·관리비가 연 3억~5억 원 줄 수 있지만, 생산성이 5%만 하락해도 연 인건비 50억 원 조직에서는 2.5억 원 손실이 발생해 절감 효과가 상당 부분 상쇄됩니다. 반대로 개발·기획처럼 독립 업무 비중이 높고, 성과지표(KPI) 측정이 가능한 조직은 생산성 저하 없이 채용 풀 확대와 이직률 감소로 순이익이 날 가능성이 큽니다. 결론적으로 “전면 도입”은 전사 일괄 적용보다, 3~6개월 파일럿으로 생산성·이직률·사무비 절감 데이터를 확인한 뒤 직무별로 확대하는 것이 비용 대비 기대수익이 가장 높습니다.


---

## 5. 구조화된 출력 (Structured Output)

LLM의 출력을 **JSON 등 특정 형식으로 강제**합니다.  
후처리(파싱, DB 저장, API 응답)가 필요할 때 사용합니다.

```
일반 출력: 자연어 문장 → 파싱 불가
구조화:   JSON/YAML → 자동 파싱 가능 → 시스템 연동 가능
```

In [6]:
# ── 구조화된 출력: JSON 형식 강제 ────────────────────────────
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", """반드시 아래 JSON 형식으로만 답변하세요. 다른 텍스트는 절대 포함하지 마세요.

  - 출력형식
  "topic": "주제",
  "pros": ["찬성 근거1", "찬성 근거2", "찬성 근거3"],
  "cons": ["반대 근거1", "반대 근거2"],
  "recommendation": "최종 권고사항",
  "confidence": 확신도(1~10)
"""),
    ("human", "다음 주제를 분석해줘: {question}")
])

# 체인 생성 및 실행
chain = structured_prompt | llm | StrOutputParser()
raw_output = chain.invoke({"question": "재택근무 전면 도입"})

print("=== LLM 원본 출력 ===")
print(raw_output)

# JSON 파싱
print("\n=== 파싱 후 활용 ===")
try:
    data = json.loads(raw_output)
    print(f"주제     : {data['topic']}")
    print(f"확신도   : {data['confidence']}/10")
    print(f"권고사항 : {data['recommendation']}")
    print(f"찬성 근거: {', '.join(data['pros'])}")
except json.JSONDecodeError as e:
    print(f"파싱 실패: {e}")
    print("→ 팁: 모델이 JSON을 못 지키면 출력에서 JSON 부분만 추출하거나 재시도 로직을 추가하세요.")

=== LLM 원본 출력 ===
{"topic":"재택근무 전면 도입","pros":["출퇴근 시간 절감으로 직원 만족도와 생산성이 향상될 수 있다","사무실 운영비와 관리비를 절감해 기업의 고정비 부담을 낮출 수 있다","지역 제약 없이 인재를 채용할 수 있어 인력 확보와 다양성 확대에 유리하다"],"cons":["대면 협업 감소로 의사소통 오류와 조직문화 약화가 발생할 수 있다","직무 특성이나 개인별 환경 차이로 인해 업무 효율과 공정성 문제가 생길 수 있다"],"recommendation":"재택근무 전면 도입은 직무 특성과 조직 성숙도를 고려해 단계적으로 추진하는 것이 바람직하며, 전면 시행 전 협업 도구, 성과관리 기준, 보안 체계를 충분히 갖추는 것을 권고한다.","confidence":9}

=== 파싱 후 활용 ===
주제     : 재택근무 전면 도입
확신도   : 9/10
권고사항 : 재택근무 전면 도입은 직무 특성과 조직 성숙도를 고려해 단계적으로 추진하는 것이 바람직하며, 전면 시행 전 협업 도구, 성과관리 기준, 보안 체계를 충분히 갖추는 것을 권고한다.
찬성 근거: 출퇴근 시간 절감으로 직원 만족도와 생산성이 향상될 수 있다, 사무실 운영비와 관리비를 절감해 기업의 고정비 부담을 낮출 수 있다, 지역 제약 없이 인재를 채용할 수 있어 인력 확보와 다양성 확대에 유리하다


---

## 6. 프롬프트 템플릿 (재사용 가능한 팩토리)

**변수화된 프롬프트**를 만들어 다양한 도메인에 재사용합니다.  
같은 구조의 프롬프트를 반복해서 만들 때 효율적입니다.

```python
# 나쁜 예: 비슷한 프롬프트를 계속 복붙
legal_prompt   = "당신은 변호사입니다... {question}"
medical_prompt = "당신은 의사입니다... {question}"

# 좋은 예: 팩토리 함수로 생성
make_expert_chain(domain="법률", style="간결하게")
make_expert_chain(domain="의료", style="쉽게 설명")
```

In [7]:
# ── 프롬프트 템플릿 팩토리 ────────────────────────────────────
def make_analyst_chain(domain: str, perspective: str, output_format: str):
    """
    재사용 가능한 분석 체인을 생성한다.

    Args:
        domain      : 전문 도메인 (예: '경영', '기술', '심리')
        perspective : 분석 관점 (예: '비용 중심', '사람 중심')
        output_format: 출력 형식 지시문
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"""당신은 {domain} 분야 전문 분석가입니다.
{perspective} 관점에서 분석하세요.

출력 형식:
{output_format}"""),
        ("human", "{question}")
    ])
    
    # 체인 생성
    chain = prompt | llm | StrOutputParser()
    return chain


# 경영 분석 체인
biz_chain = make_analyst_chain(
    domain="경영",
    perspective="ROI와 생산성 중심",
    output_format="[핵심 지표] → [예상 효과] → [실행 권고] (각 2~3줄)"
)

# 조직심리 분석 체인
psych_chain = make_analyst_chain(
    domain="조직심리",
    perspective="구성원 동기·몰입 중심",
    output_format="[심리적 영향] → [리스크] → [권고사항] (각 2~3줄)"
)

question = "재택근무 전면 도입, 찬성해야 할까요?"

print("=== 경영 분석 관점 ===")
print(biz_chain.invoke({"question": question}))

print("\n=== 조직심리 관점 ===")
print(psych_chain.invoke({"question": question}))

=== 경영 분석 관점 ===
[핵심 지표]  
고정비 절감(사무실 임차료·관리비), 1인당 생산성, 이직률, 채용 경쟁력이 핵심입니다.  
재택근무 전면 도입 시 비용 구조는 가벼워질 가능성이 크지만, 협업 밀도·관리 난이도·보안 리스크가 동시에 KPI에 반영됩니다.  

[예상 효과]  
ROI 관점에서는 사무공간 축소와 채용 풀 확대를 통해 중장기 수익성이 개선될 수 있습니다.  
반면 신입 온보딩 저하, 의사결정 지연, 조직문화 약화로 인해 생산성 편차가 커지면 절감 비용 이상으로 성과 손실이 발생할 수 있습니다.  

[실행 권고]  
전면 찬성보다는 직무별로 구분해 단계 도입하는 것이 합리적입니다.  
개인 집중형 업무는 재택 중심, 협업·교육·보안 민감 직무는 하이브리드 또는 출근 유지로 설계하고, 3~6개월간 생산성·이직률·성과지표를 측정해 확대 여부를 결정하는 것을 권고합니다.

=== 조직심리 관점 ===
[심리적 영향]  
재택근무 전면 도입은 자율성·통제감을 높여 동기와 직무만족을 끌어올릴 수 있습니다. 반면 소속감, 비공식 상호작용, 즉각적 피드백이 약해지면 몰입이 개인차에 따라 크게 흔들릴 수 있습니다.  

[리스크]  
구성원 간 자기조절 역량, 주거환경, 디지털 협업 역량 차이로 인해 성과와 심리적 부담의 격차가 커질 수 있습니다. 특히 신입, 저성과 회복자, 관계 기반으로 배우는 직무는 고립감·학습지연·조직정체성 약화 위험이 큽니다.  

[권고사항]  
전면 찬반보다 직무·숙련도·팀 상호의존성 기준으로 하이브리드 또는 선택형 모델을 우선 검토하는 것이 바람직합니다. 재택을 확대하더라도 목표·피드백 체계, 정기적 대면 접점, 관리자 코칭, 온보딩 보완장치를 함께 설계해야 몰입 저하를 막을 수 있습니다.


---

## 7. 비교 실험: 같은 질문, 다른 기법

지금까지 배운 6가지 기법을 **같은 질문**에 적용해 결과를 비교합니다.  
답변의 구조, 깊이, 형식이 얼마나 다른지 확인하세요.

In [8]:
# ── 비교 실험: 기법별 답변 길이 & 구조 비교 ──────────────────
QUESTION = "재택근무 전면 도입, 찬성해야 할까요?"

# 각 기법의 체인 목록
techniques = [
    ("Zero-shot",         zero_shot_prompt),
    ("Few-shot",          few_shot_prompt),
    ("Chain-of-Thought",  cot_prompt),
]

print(f"질문: {QUESTION}")
print("=" * 60)

results = {}
for name, prompt in techniques:
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": QUESTION})
    results[name] = answer

# 결과 요약 출력
print(f"\n{'기법':<20} {'글자수':>8} {'줄 수':>6}")
print("-" * 38)
for name, answer in results.items():
    print(f"{name:<20} {len(answer):>8,} {len(answer.splitlines()):>6}")

질문: 재택근무 전면 도입, 찬성해야 할까요?

기법                        글자수    줄 수
--------------------------------------
Zero-shot                 916     43
Few-shot                  215     12
Chain-of-Thought          873     27


In [9]:
# ── 답변 전체 출력 ────────────────────────────────────────────
for name, answer in results.items():
    print(f"\n{'='*60}")
    print(f"[{name}]")
    print('='*60)
    # 처음 400자만 출력 (너무 길면 ...으로 줄임)
    preview = answer[:400]
    print(preview)
    if len(answer) > 400:
        print(f"\n... (총 {len(answer):,}자 중 400자 미리보기)")


[Zero-shot]
재택근무 전면 도입은 **무조건 찬성/반대**로 보기보다, **업무 성격과 조직 문화에 따라 신중히 판단해야 하는 사안**입니다. 그래도 질문이 “찬성해야 할까요?”라면, 제 답은:

**“조건부 찬성”**입니다.

### 찬성하는 이유
1. **출퇴근 시간 절약**
   - 직원 피로가 줄고 삶의 질이 올라갑니다.
   - 절약된 시간을 업무나 휴식에 더 효율적으로 쓸 수 있습니다.

2. **업무 집중도 향상 가능**
   - 사무실의 불필요한 회의나 방해 요소가 줄어들 수 있습니다.
   - 특히 개인 단위의 기획, 개발, 문서 작업은 재택이 더 효율적일 수 있습니다.

3. **기업 비용 절감**
   - 사무실 임대료, 관리비, 비품비 등을 줄일 수 있습니다.

4. **인재 채용 범위 확대**
 

... (총 916자 중 400자 미리보기)

[Few-shot]
답변:
✅ 찬성 근거
- 출퇴근 시간 절감으로 업무 만족도와 효율 향상
- 지역 제약 없이 우수 인재 채용 가능
- 사무실 임대·운영 비용 절감

❌ 반대 근거
- 팀 협업과 소통의 질 저하 가능성
- 직원 고립감 및 조직문화 약화 우려
- 보안 관리와 성과 관리가 더 어려워질 수 있음

💡 결론: 전면 도입보다는 직무 특성에 따라 하이브리드 근무를 우선 검토하는 것이 현실적입니다.

[Chain-of-Thought]
[1단계: 문제 파악]  
질문은 “재택근무를 전면 도입하는 것에 찬성할지”에 대한 판단입니다. 즉, 일부 허용이 아니라 **모든 구성원 또는 대부분의 업무를 재택 중심으로 전환하는 정책**이 바람직한지 묻고 있습니다.

[2단계: 이해관계자 분석]  
영향을 받는 주체는 다음과 같습니다.

- **직원**: 출퇴근 부담, 업무 집중도, 워라밸, 고립감 등에 영향
- **기업/경영진**: 생산성, 조직관리, 비용 절감, 성과평가 방식 변화
- **팀장/중간관리자**: 협업 관리, 소통 방식, 성과관리 부담 증가
- **고객/거래처**: 대응 속도, 서비

---

## 핵심 정리

```python
# 프롬프트 엔지니어링 선택 가이드

# 1. Zero-shot: 빠른 테스트, 일반 질문
chain = ChatPromptTemplate.from_messages([("human", "{q}")]) | llm | StrOutputParser()

# 2. Few-shot: 특정 형식 유도, 스타일 일관성
#    → 시스템 프롬프트에 예시 2~3개 포함

# 3. Chain-of-Thought: 복잡한 추론, 다단계 분석
#    → "단계별로 생각하세요" 지시문 추가

# 4. 역할 프롬프트: 전문 도메인, 특정 관점 필요 시
#    → 구체적일수록 좋음 ("전문가" < "10년 경력 변호사")

# 5. 구조화 출력: 파싱·연동이 필요한 경우
#    → JSON 형식 + "다른 텍스트 포함 금지" 명시

# 6. 팩토리 패턴: 동일 구조 반복 시
def make_chain(domain, style):
    prompt = ChatPromptTemplate.from_messages([...])
    return prompt | llm | StrOutputParser()
```

| 기법 | 최적 상황 | 주의사항 |
|---|---|---|
| Zero-shot | 범용 질문, 빠른 프로토타입 | 형식 보장 없음 |
| Few-shot | 특정 형식·스타일 필요 | 예시가 너무 많으면 토큰 낭비 |
| CoT | 복잡한 추론·판단 | 응답이 길어짐 |
| 역할 | 전문 도메인 | 구체적일수록 효과적 |
| 구조화 출력 | API 연동, 파싱 필요 | 100% 보장 안 됨 (검증 로직 필요) |
| 팩토리 | 반복 사용 패턴 | 과도한 추상화 주의 |
